# Ликвидность: разность разностей

Проверка предпосылки параллельных трендов, оценка устойчивого эффекта и
отдельное измерение всплеска в дни самой ребалансировки.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd

from src import moex, pipeline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
P = moex.DATA_PROCESSED

## Контрольная группа и баланс ковариат

In [2]:
from src import matching
matches = pd.read_parquet(P / 'matches.parquet')
print(matches.match_quality.value_counts().to_string())
print(f'\nмедианное расстояние: {matches.distance.median():.3f}, максимум: {matches.distance.max():.3f}')
matching.covariate_balance(matches).round(3)

match_quality
та же отрасль, близкое соответствие     17
другая отрасль, близкое соответствие     9

медианное расстояние: 0.600, максимум: 0.962


,признак,среднее в тесте,среднее в контроле,стандартизованная разность
0,логарифм капитализации,25.607,25.232,0.406
1,логарифм числа сделок,9.582,9.217,0.511


Стандартизованные разности выше общепринятого ориентира 0.25. Это
неустранимо: в индекс отбирают именно по размеру и ликвидности, поэтому
контроля «такой же, но не в индексе» не существует. Разность разностей
требует параллельности трендов, а не равенства уровней.

## Параллельные тренды

In [3]:
trends = pd.read_parquet(P / 'parallel_trends.parquet')
trends[['event_type', 'metric_name', 'f_stat', 'p_value', 'n']].round(4)

,event_type,metric_name,f_stat,p_value,n
0,включение,логарифм оборота в рублях,0.4915,0.7420,1680
1,включение,логарифм числа сделок,0.6014,0.6648,1680
2,включение,прокси спреда (HIGH-LOW)/CLOSE,2.0128,0.1199,1680
3,включение,день без торгов,1.0389,0.3671,1700
4,исключение,логарифм оборота в рублях,0.7677,0.5587,1200
5,исключение,логарифм числа сделок,0.8238,0.5254,1200
6,исключение,прокси спреда (HIGH-LOW)/CLOSE,0.5040,0.7332,1200
7,исключение,день без торгов,NaN,NaN,1200


Предпосылка не отвергается ни по одной метрике. Оговорка: мощность теста
на таком числе кластеров невелика, и неотвержение — слабое утверждение.

## Устойчивый эффект на ликвидность

In [4]:
did_results = pd.read_parquet(P / 'did_results.parquet')
columns = [c for c in ['event_type', 'metric_name', 'coefficient', 'se',
                       'p_value', 'p_holm', 'n', 'n_clusters']
           if c in did_results.columns]
did_results[columns].round(4)

,event_type,metric_name,coefficient,se,p_value,p_holm,n,n_clusters
0,включение,логарифм оборота в рублях,0.1069,0.1731,0.5370,1.0,3380,29
1,включение,логарифм числа сделок,0.1801,0.1326,0.1743,1.0,3380,29
2,включение,прокси спреда (HIGH-LOW)/CLOSE,0.0012,0.0032,0.6981,1.0,3380,29
3,включение,день без торгов,-0.0235,0.0213,0.2696,1.0,3400,29
4,исключение,логарифм оборота в рублях,0.0386,0.2028,0.8492,1.0,2400,21
5,исключение,логарифм числа сделок,0.1323,0.1344,0.3250,1.0,2400,21
6,исключение,прокси спреда (HIGH-LOW)/CLOSE,0.0064,0.0047,0.1680,1.0,2400,21
7,исключение,день без торгов,0.0000,0.0000,NaN,NaN,2400,21


Ни одна оценка не приближается к значимости. Ожидание, что индексные фонды
надолго поддерживают повышенную ликвидность бумаги, данными не
подтверждается.

## Всплеск в дни ребалансировки

Основная оценка намеренно исключает окрестность события. Но сама сделка
фондов наблюдаема, и её стоит измерить отдельно — иначе механизм остаётся
непроверенным утверждением.

In [5]:
from src import did
panel = pd.read_parquet(P / 'liquidity_panel.parquet')

for metric, name in [('log_value', 'оборот в рублях'), ('log_trades', 'число сделок')]:
    spike = did.rebalancing_spike(panel, metric)
    print(f'--- {name} ---')
    for _, row in spike.iterrows():
        print(f"  {row['event_type']:11s} тест {row['treated_change']:+.0%}, "
              f"контроль {row['control_change']:+.0%}, "
              f"разность разностей {row['difference']:+.0%}")
    print()

--- оборот в рублях ---
  включение   тест +103%, контроль +19%, разность разностей +70%
  исключение  тест +19%, контроль -29%, разность разностей +68%

--- число сделок ---
  включение   тест +58%, контроль +15%, разность разностей +37%
  исключение  тест +31%, контроль -29%, разность разностей +85%



Всплеск симметричен: при исключении фонды продают, и оборот растёт так же,
как при покупке. Механизм работает в обе стороны.